# SOLA with Sobolev H² Model Space and Straight-Ray Data

This notebook demonstrates the **SOLA** (Subtractive Optimally Localised Averages) method using:

- A **Sobolev H²** model space on the unit cube `[0,1]³`
- **Straight-ray path-integral** measurements connecting seismic sources to surface stations

The workflow follows `sola_noise.ipynb` but replaces the Lebesgue data functionals with `PathIntegralFunctional` objects derived from three-dimensional straight rays.


In [ ]:
import sys
import time
from pathlib import Path

# ── Robust bootstrap — finds workspace root regardless of kernel CWD ──
def _find_workspace_root(start: Path) -> Path:
    """Walk upward until the directory containing pygeoinf and pygeoinf3D is found."""
    p = start.resolve()
    for _ in range(8):
        if (p / "pygeoinf").is_dir() and (p / "pygeoinf3D").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError(f"Cannot find workspace root from {start}")

WORKSPACE_ROOT = _find_workspace_root(Path.cwd())
for _repo in [WORKSPACE_ROOT / "pygeoinf", WORKSPACE_ROOT / "pygeoinf3D"]:
    _s = str(_repo)
    if _repo.exists() and _s not in sys.path:
        sys.path.insert(0, _s)

import numpy as np
from scipy.interpolate import RegularGridInterpolator
import plotly.graph_objects as go

from pygeoinf3d import BoxDomain, Lebesgue3D, Function3D
from pygeoinf3d.providers.tensor_product import BoxLaplacianSpectrumProvider
from pygeoinf3d.operators.bessel_sobolev_box import BesselSobolevBox
from pygeoinf3d.spaces.sobolev import Sobolev3D
from pygeoinf3d.spaces.forms import PathIntegralFunctional, VolumeKernel3D
from pygeoinf3d.core.paths import Path3D
from pygeoinf3d.providers.function_providers import GaussianProvider3D
from pygeoinf3d.plot import (
    plot_function3d_axis_slider,
    compare_function3d_axis_sliders,
    compare_panels_axis_dropdown_sliders,
)

print("Imports OK")
print(f"Workspace root: {WORKSPACE_ROOT}")


## 1. Sobolev H² Model Space

We build a Sobolev space on `[0,1]³` using a Bessel–Sobolev weight operator with order `s=2`. The spectral representation uses a tensor-product Laplacian with `8×8×8 = 512` Neumann eigenmodes.

In [ ]:
np.random.seed(42)
box = BoxDomain(np.zeros(3), np.ones(3))
base_space = Lebesgue3D(box)

spec_provider = BoxLaplacianSpectrumProvider(
    box, truncation=(8, 8, 8), bc_type="neumann"
)
W     = BesselSobolevBox(base_space, spec_provider, shift=1.0, order=2.0)
W_inv = BesselSobolevBox(base_space, spec_provider, shift=1.0, order=-2.0)
sobolev = Sobolev3D(base_space, W, W_inv)

print("Box     : [0,1]^3")
print("Trunc.  : (8,8,8) → 512 modes")
print("Sobolev : H² (Bessel–Sobolev shift=1.0, order=2.0)")


## 2. Ray Geometry

We place **10 seismic stations** randomly on the `z=1` surface and **30 seismic sources** uniformly at random inside the volume. Each source is assigned to one station, giving 30 straight-ray paths.

In [ ]:
N_stations = 10
N_d = 30  # number of sources / rays

np.random.seed(42)
station_xy = np.random.uniform(0.05, 0.95, size=(N_stations, 2))
stations = np.column_stack([station_xy, np.ones(N_stations)])

sources = np.random.uniform(0.05, 0.95, size=(N_d, 3))
station_idx = np.random.randint(0, N_stations, size=N_d)

paths = [
    Path3D.from_polyline(np.array([sources[i], stations[station_idx[i]]]))
    for i in range(N_d)
]

print(f"Stations : {N_stations} at z=1")
print(f"Sources  : {N_d} random in [0.05,0.95]^3")
print(f"Rays     : {len(paths)}")

# --- Visualise geometry ---
_tgx = np.linspace(0.1, 0.9, 3)
target_grid_preview = np.array([[x, y, z] for x in _tgx for y in _tgx for z in _tgx])

fig_geo = go.Figure()
for i in range(N_d):
    src, sta = sources[i], stations[station_idx[i]]
    fig_geo.add_trace(go.Scatter3d(
        x=[src[0], sta[0]], y=[src[1], sta[1]], z=[src[2], sta[2]],
        mode="lines", line=dict(color="steelblue", width=2),
        showlegend=(i == 0), name="Rays" if i == 0 else None, legendgroup="rays",
    ))
fig_geo.add_trace(go.Scatter3d(
    x=stations[:, 0], y=stations[:, 1], z=stations[:, 2],
    mode="markers", marker=dict(color="red", size=7, symbol="square"), name="Stations",
))
fig_geo.add_trace(go.Scatter3d(
    x=sources[:, 0], y=sources[:, 1], z=sources[:, 2],
    mode="markers", marker=dict(color="limegreen", size=4, symbol="circle"), name="Sources",
))
fig_geo.add_trace(go.Scatter3d(
    x=target_grid_preview[:, 0], y=target_grid_preview[:, 1], z=target_grid_preview[:, 2],
    mode="markers", marker=dict(color="orange", size=4, symbol="diamond"), name="Targets",
))
fig_geo.update_layout(
    title="Ray geometry",
    scene=dict(
        xaxis_title="x", yaxis_title="y", zaxis_title="z",
        xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1]), zaxis=dict(range=[0, 1]),
    ),
)
fig_geo.show()


## 3. Forward Forms, True Model, and Synthetic Data

Each `PathIntegralFunctional` maps a Sobolev function to its line integral along the corresponding ray. The true model is a superposition of three Gaussian blobs. Synthetic data are generated by evaluating the forward forms on the true model and adding 5% Gaussian noise.

In [ ]:
t0 = time.time()
forms_G = [PathIntegralFunctional(sobolev, p, n_quad=200) for p in paths]
print(f"Built {len(forms_G)} PathIntegralFunctional on Sobolev3D")

# True model: 3 Gaussian blobs (same as sola_noise.py)
def _true_model_callable(x):
    if np.ndim(x) == 1:
        x = x[None, :]
    def _d(c): return x - c[None, :]
    return (
        + 1.5 * np.exp(-0.5 * np.sum((_d(np.array([0.30, 0.30, 0.50])) / 0.12) ** 2, axis=1))
        + 1.0 * np.exp(-0.5 * np.sum((_d(np.array([0.70, 0.65, 0.55])) / 0.10) ** 2, axis=1))
        - 0.8 * np.exp(-0.5 * np.sum((_d(np.array([0.55, 0.40, 0.45])) / 0.08) ** 2, axis=1))
    ).squeeze()

m_true = Function3D(box, evaluate_callable=_true_model_callable, name="true model")

d_bar = np.array([f(m_true) for f in forms_G])
noise_std = 0.05 * np.max(np.abs(d_bar))
C_D = noise_std ** 2 * np.eye(N_d)
rng = np.random.default_rng(123)
d_tilde = d_bar + rng.normal(0, noise_std, size=N_d)

snr = np.linalg.norm(d_bar) / np.linalg.norm(d_tilde - d_bar)
print(f"Noise std : {noise_std:.4e}")
print(f"SNR       : {snr:.1f}")
print(f"Elapsed   : {time.time()-t0:.1f}s")

plot_function3d_axis_slider(
    m_true, axis="z", n_slices=10, resolution=50, title="True model (z-slider)",
    show_plot=True,
).show()


## 4. SOLA Matrix Construction

The SOLA weights are found by solving

$$X = C_{TG}\,(C_{GG} + C_D)^{-1}$$

where:
- $C_{GG}[i,j] = \ell_i(r_j)$ is the Gram matrix of forward representers in the Sobolev inner product,
- $C_{TG}[k,i] = \ell_{T_k}(r_i)$ is the cross-Gram matrix of target and forward representers,
- $r_i = W^{-1}\ell_i^*$ is the Sobolev representer of the $i$-th forward form.

Computing the representers is the computational bottleneck (one Sobolev solve per forward functional).

In [ ]:
t0 = time.time()

# Target forms on 3×3×3 grid
target_grid = np.array([
    [x, y, z]
    for x in np.linspace(0.1, 0.9, 3)
    for y in np.linspace(0.1, 0.9, 3)
    for z in np.linspace(0.1, 0.9, 3)
])
N_p = len(target_grid)  # 27
fp_T = GaussianProvider3D(box, centers=target_grid, sigmas=0.10, normalized=True)
forms_T = [VolumeKernel3D(base_space, fp_T.get_function(k)) for k in range(N_p)]
print(f"Target forms T : {N_p} VolumeKernel3D on 3×3×3 grid")

# Sobolev representers r_i = W^{-1} l_i^*
print(f"Computing {N_d} representers...")
representers = []
for i, f in enumerate(forms_G):
    representers.append(sobolev.from_dual(f))
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{N_d}  {time.time()-t0:.0f}s")
print(f"Representers done [{time.time()-t0:.1f}s]")

# Gram matrix C_GG[i,j] = l_i(r_j)
C_GG = np.zeros((N_d, N_d))
for i in range(N_d):
    for j in range(i, N_d):
        v = forms_G[i](representers[j])
        C_GG[i, j] = v
        C_GG[j, i] = v

# Cross-Gram C_TG[k,i] = l_T_k(r_i)
C_TG = np.zeros((N_p, N_d))
for k in range(N_p):
    for i in range(N_d):
        C_TG[k, i] = forms_T[k](representers[i])

# SOLA weights X = C_TG @ (C_GG + C_D)^{-1}
A = C_GG + C_D
X = C_TG @ np.linalg.solve(A, np.eye(N_d))

print(f"X shape: {X.shape}")
print(f"Condition number: {np.linalg.cond(A):.2e}")
print(f"Total elapsed: {time.time()-t0:.1f}s")


## 5. Property Inference

Apply the SOLA weights to the noisy data:

$$\hat{p}_k = \sum_i X_{ki}\,\tilde{d}_i, \qquad \Sigma_p = X\,C_D\,X^\top$$

In [ ]:
p_mean = X @ d_tilde
p_cov  = X @ C_D @ X.T
p_std  = np.sqrt(np.diag(p_cov))
p_true = np.array([forms_T[k](m_true) for k in range(N_p)])

print(f"{'k':>3} | {'cx':>5} {'cy':>5} {'cz':>5} | {'true':>8} | {'mean':>8} | {'2σ':>8}")
print('-' * 55)
for k in range(N_p):
    cx, cy, cz = target_grid[k]
    print(f"{k:3d} | {cx:.2f}  {cy:.2f}  {cz:.2f} | {p_true[k]:8.4f} | {p_mean[k]:8.4f} | {2*p_std[k]:8.4f}")

# Interpolate 27-point grid values to Function3D fields
_grid_ax = np.unique(target_grid[:, 0])

def _make_prop_fn(values, name=''):
    grid_vals = values.reshape(3, 3, 3)
    interp = RegularGridInterpolator(
        (_grid_ax, _grid_ax, _grid_ax), grid_vals,
        method='linear', bounds_error=False, fill_value=0.0,
    )
    def _fn(x): return interp(np.clip(np.atleast_2d(x), _grid_ax[0], _grid_ax[-1])).squeeze()
    return Function3D(box, evaluate_callable=_fn, name=name)

p_mean_field  = _make_prop_fn(p_mean,     'SOLA mean')
p_2std_field  = _make_prop_fn(2 * p_std,  'Uncertainty 2σ')

compare_function3d_axis_sliders(
    [p_mean_field, p_2std_field], axis='z', n_slices=10, resolution=50,
    titles=['SOLA estimate (mean)', 'Uncertainty (2σ)'], show_plot=True,
).show()


## 6. Unimodularity Constraint

A rank-1 correction ensures $X_{\text{uni}}\,G(\mathbf{1}) = \mathbf{1}$ (each averaging kernel integrates to 1). The correction uses the data-space metric $(C_{GG}+C_D)$ to minimise the perturbation to $X$:

$$X_{\text{uni}} = X - \frac{(X v - w)}{v^\top u}\,u^\top, \qquad u = (C_{GG}+C_D)^{-1}v, \quad v = G(\mathbf{1}), \quad w = \mathbf{1}_{N_p}$$

In [ ]:
ones_fn = Function3D(
    box,
    evaluate_callable=lambda x: np.ones(np.atleast_2d(x).shape[0]),
    name="ones",
)
v = np.array([f(ones_fn) for f in forms_G])   # G(1)
w = np.ones(N_p)                               # desired unimodular values

# Rank-1 correction using the data-space metric (C_GG + C_D)
u    = np.linalg.solve(C_GG + C_D, v)          # (C_GG + C_D)^{-1} v
beta = float(np.dot(v, u))
X_uni = X - np.outer((X @ v - w) / beta, u)

check = X_uni @ v
print('X_uni(G 1) — should equal ones (first 5):', np.asarray(check).round(4)[:5])

p_mean_uni    = X_uni @ d_tilde
P_cov_uni     = X_uni @ C_D @ X_uni.T
p_std_uni     = np.sqrt(np.diag(P_cov_uni))
p_mean_uni_field = _make_prop_fn(p_mean_uni,     'Unimodular mean')
p_2std_uni_field = _make_prop_fn(2 * p_std_uni,  'Unimodular 2σ')

compare_function3d_axis_sliders(
    [p_mean_field, p_mean_uni_field], axis='z', n_slices=10, resolution=50,
    titles=['Mean — unconstrained', 'Mean — unimodular'], show_plot=True,
).show()

compare_function3d_axis_sliders(
    [p_2std_field, p_2std_uni_field], axis='z', n_slices=10, resolution=50,
    titles=['2σ — unconstrained', '2σ — unimodular'], show_plot=True,
).show()


## 7. Resolving Kernels

The resolving kernel for target $k$ is

$$A_k(x) = \sum_i (X_{\text{uni}})_{ki}\, r_i(x)$$

Ideally $A_k$ is concentrated near the $k$-th target Gaussian. We compare resolving kernels to their target Gaussians for a selection of 5 target locations.

In [ ]:
PREVIEW_IDX = [0, 6, 13, 20, 26]

def make_resolving_kernel(k, X_mat, reps, bx):
    weights = X_mat[k, :]
    def _eval(x):
        pts = np.asarray(x, dtype=float)
        single = pts.ndim == 1
        pts = pts[None, :] if single else pts
        result = sum(w * r(pts) for w, r in zip(weights, reps))
        return float(result[0]) if single else result
    return Function3D(bx, evaluate_callable=_eval, name=f'RK k={k}')

rk_fns  = [make_resolving_kernel(k, X_uni, representers, box) for k in PREVIEW_IDX]
tgt_fns = [fp_T.get_function(k) for k in PREVIEW_IDX]

rk_labels  = [f'RK k={k}  ({target_grid[k,0]:.2f},{target_grid[k,1]:.2f},{target_grid[k,2]:.2f})' for k in PREVIEW_IDX]
tgt_labels = [f'Target k={k}  ({target_grid[k,0]:.2f},{target_grid[k,1]:.2f},{target_grid[k,2]:.2f})' for k in PREVIEW_IDX]

compare_panels_axis_dropdown_sliders(
    panel_functions=[rk_fns, tgt_fns],
    panel_labels=[rk_labels, tgt_labels],
    panel_titles=['Resolving kernels', 'Target Gaussians'],
    axis='z', n_slices=10, resolution=40, show_plot=True,
).show()
